# BuyerBench Results Analysis

This notebook loads `results/experiments/FULL-REPORT.json` and produces four publication-quality
visualisations:

1. **Radar chart** — per-agent score profile across all three pillars
2. **Bar chart** — Bias Susceptibility Index (BSI) by bias type and agent
3. **Heatmap** — compliance adherence rate by scenario × agent
4. **Boxplot** — latency distribution by agent and mode

Run `python -m buyerbench report --experiment-dir results/experiments/` first to generate
the report JSON.

In [ ]:
import json
import pathlib

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
import seaborn as sns

# ── Resolve report path relative to this notebook ────────────────────────────
NOTEBOOK_DIR = pathlib.Path().resolve()
REPORT_PATH = NOTEBOOK_DIR.parent / "results" / "experiments" / "FULL-REPORT.json"

if not REPORT_PATH.exists():
    raise FileNotFoundError(
        f"Report not found at {REPORT_PATH}\n"
        "Run: python -m buyerbench report --experiment-dir results/experiments/"
    )

with REPORT_PATH.open() as fh:
    report = json.load(fh)

print(f"Loaded report from {REPORT_PATH}")
print(f"Generated at: {report.get('generated_at')}")
print(f"Per-pillar rows: {len(report.get('per_pillar_aggregate', []))}")
print(f"BSI rows:        {len(report.get('bias_susceptibility_table', []))}")
print(f"Security rows:   {len(report.get('security_violation_table', []))}")

In [ ]:
# ── Shared style ──────────────────────────────────────────────────────────────
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 150})
PILLAR_ORDER = ["PILLAR1", "PILLAR2", "PILLAR3"]
PILLAR_LABELS = ["Pillar 1\nCapability", "Pillar 2\nEconomics", "Pillar 3\nSecurity"]

## 1. Radar Chart — Agent Score Profile Across Pillars

Each agent is represented as a polygon on a three-axis radar chart.
Agents that score uniformly high form a large, filled triangle.
Gaps indicate which pillar needs improvement.

In [ ]:
agg_df = pd.DataFrame(report.get("per_pillar_aggregate", []))

if agg_df.empty:
    print("No aggregate data — all agents were skipped. Radar chart omitted.")
else:
    # Pivot: rows = agent, columns = pillar, values = mean_score
    pivot = (
        agg_df.pivot_table(index="agent_id", columns="pillar", values="mean_score", aggfunc="mean")
        .reindex(columns=PILLAR_ORDER, fill_value=0.0)
    )

    n_axes = len(PILLAR_ORDER)
    angles = np.linspace(0, 2 * np.pi, n_axes, endpoint=False).tolist()
    angles += angles[:1]  # close the polygon

    fig, ax = plt.subplots(figsize=(6, 6), subplot_kw={"polar": True})
    cmap = plt.get_cmap("tab10")

    for idx, (agent_id, row) in enumerate(pivot.iterrows()):
        values = row.tolist() + row.tolist()[:1]
        color = cmap(idx % 10)
        ax.plot(angles, values, "o-", linewidth=1.8, color=color, label=agent_id)
        ax.fill(angles, values, alpha=0.12, color=color)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(PILLAR_LABELS, fontsize=10)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=7)
    ax.set_title("Agent Score Profile Across Pillars", fontsize=13, pad=20)
    ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.15), fontsize=8)

    plt.tight_layout()
    plt.savefig("radar_agent_pillars.png", bbox_inches="tight")
    plt.show()
    print("Saved: radar_agent_pillars.png")

## 2. Bar Chart — Bias Susceptibility Index (BSI) by Bias Type and Agent

Higher BSI means the agent changed its decision when only the presentation changed —
indicating susceptibility to that bias. Ideal BSI is 0.

In [ ]:
bsi_df = pd.DataFrame(report.get("bias_susceptibility_table", []))

if bsi_df.empty:
    print("No BSI data available — all agents were skipped. Bar chart omitted.")
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.barplot(
        data=bsi_df,
        x="bias_type",
        y="bsi",
        hue="agent_id",
        ax=ax,
        palette="muted",
        errorbar="sd",
    )
    ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
    ax.set_ylim(0, 1)
    ax.set_xlabel("Bias Type", fontsize=11)
    ax.set_ylabel("Bias Susceptibility Index", fontsize=11)
    ax.set_title("BSI by Bias Type and Agent", fontsize=13)
    ax.tick_params(axis="x", rotation=30)
    ax.legend(title="Agent", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)

    plt.tight_layout()
    plt.savefig("bsi_by_bias_type.png", bbox_inches="tight")
    plt.show()
    print("Saved: bsi_by_bias_type.png")

## 3. Heatmap — Compliance Adherence Rate by Scenario × Agent

Each cell shows how often an agent followed security/compliance rules for that scenario.
1.0 = fully compliant, 0.0 = all rules violated.

In [ ]:
sec_df = pd.DataFrame(report.get("security_violation_table", []))

if sec_df.empty:
    print("No security data available — all agents were skipped. Heatmap omitted.")
else:
    pivot_sec = sec_df.pivot_table(
        index="scenario_id",
        columns="agent_id",
        values="compliance_adherence_rate",
        aggfunc="mean",
    )

    fig, ax = plt.subplots(figsize=(max(6, len(pivot_sec.columns) * 1.5), max(4, len(pivot_sec) * 0.8)))
    sns.heatmap(
        pivot_sec,
        annot=True,
        fmt=".2f",
        vmin=0,
        vmax=1,
        cmap="RdYlGn",
        linewidths=0.5,
        ax=ax,
        cbar_kws={"label": "Compliance Adherence Rate"},
    )
    ax.set_xlabel("Agent", fontsize=11)
    ax.set_ylabel("Scenario", fontsize=11)
    ax.set_title("Compliance Adherence Rate by Scenario × Agent", fontsize=13)
    ax.tick_params(axis="x", rotation=30)

    plt.tight_layout()
    plt.savefig("compliance_heatmap.png", bbox_inches="tight")
    plt.show()
    print("Saved: compliance_heatmap.png")

## 4. Boxplot — Latency Distribution by Agent and Mode

Shows how response latency varies per agent configuration.
The latency data comes from per-scenario result JSONs (field: `latency_ms`).
If no latency data is recorded the plot is omitted.

In [ ]:
# Collect latency from all raw per-scenario result JSONs in the experiment dir.
# generate_full_report drops per-response latency, so we re-scan the JSON files.
import os

EXP_DIR = REPORT_PATH.parent
latency_records = []

for pillar_dir in sorted(EXP_DIR.iterdir()):
    if not (pillar_dir.is_dir() and pillar_dir.name.startswith("pillar")):
        continue
    for agent_dir in sorted(pillar_dir.iterdir()):
        if not agent_dir.is_dir():
            continue
        agent_id = agent_dir.name
        # derive mode from last segment of agent_id (baseline / skills / mcp)
        mode = agent_id.rsplit("-", 1)[-1] if "-" in agent_id else agent_id
        for jf in agent_dir.glob("*.json"):
            try:
                data = json.loads(jf.read_text())
            except Exception:
                continue
            if data.get("status") == "skipped":
                continue
            latency_ms = data.get("latency_ms") or data.get("response", {}).get("latency_ms")
            if latency_ms is not None:
                latency_records.append({
                    "agent_id": agent_id,
                    "mode": mode,
                    "latency_ms": float(latency_ms),
                    "pillar": pillar_dir.name.upper(),
                })

lat_df = pd.DataFrame(latency_records)

if lat_df.empty:
    print("No latency data found in result JSONs. Boxplot omitted.")
    print("(Latency is recorded only when CLI agents actually run — not for skipped/mock runs.)")
else:
    fig, ax = plt.subplots(figsize=(max(8, len(lat_df["agent_id"].unique()) * 1.4), 5))
    sns.boxplot(
        data=lat_df,
        x="agent_id",
        y="latency_ms",
        hue="mode",
        ax=ax,
        palette="Set2",
        flierprops={"marker": "o", "markersize": 3, "alpha": 0.5},
    )
    ax.set_xlabel("Agent", fontsize=11)
    ax.set_ylabel("Latency (ms)", fontsize=11)
    ax.set_title("Response Latency by Agent and Mode", fontsize=13)
    ax.tick_params(axis="x", rotation=30)
    ax.legend(title="Mode", bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8)

    plt.tight_layout()
    plt.savefig("latency_boxplot.png", bbox_inches="tight")
    plt.show()
    print("Saved: latency_boxplot.png")